# QMCI Pi estimation with native FlagQuantum
All thirteen source code cells are mapped below. Source precision, strict circle boundary and reporting formula are retained. Every executed Grover program and the QPE program is compared with an independent Circuit, including retained coordinates and absolute phase.


## Four-bit lattice
Source code cell 6.


In [ ]:
import math
from io import BytesIO

import matplotlib.pyplot as plt
from IPython.display import Image, display
from matplotlib.figure import Figure

from examples.synthesis.tutorials.basics.qmci_pi_estimation.overview.operations import (
    build_program,
    build_qpe,
)
from examples.synthesis.tutorials.basics.qmci_pi_estimation.overview.workflow import (
    grid_pi,
    plot_estimates,
    plot_lattice,
    run_iterative,
    sample_qpe,
)


def show_figure(figure: Figure) -> None:
    image = BytesIO()
    figure.savefig(image, format="png", dpi=130)
    display(Image(data=image.getvalue()))
    plt.close(figure)


show_figure(plot_lattice(4))


## Classical reference grids
Source code cell 8.


In [ ]:
for axis_length in (3, 5, 6):
    print(
        "Grid axis length:",
        axis_length,
        "points:",
        axis_length**2,
        "pi:",
        grid_pi(axis_length),
    )


## Original three-bit QPE program
Source code cell 10.


In [ ]:
phase_program = build_qpe()
print(phase_program.report["resources"])


## Actual local phase observations
Source code cell 11.


In [ ]:
phase_result = sample_qpe(1800)
print(phase_result.observations)
print("Independent full-state check:", phase_result.state_check)


## Preserve and qualify the source reporting formula
Source code cell 12.


In [ ]:
print("Source 4*sin(pi*phase):", phase_result.source_pi)
print("Squared-probability comparison:", phase_result.squared_probability_pi)
print("Exact finite lattice (8 points per axis):", grid_pi(8))
print("Continuum pi:", math.pi)


## Count-only bounded iterative estimation
Source code cell 15.


In [ ]:
print("Local example profile: Hoeffding intervals and checked quadrant inversion.")
print("This is not an implementation-equivalent reproduction of hosted Classiq IQAE.")
print("No quantum speedup or hardware timing claim is made.")


## Three-bit coordinate state preparation
Source code cell 16.


In [ ]:
preparation_program = build_program(3, 0)
print(preparation_program.report["resources"])


## Source epsilon and alpha
Source code cell 18.


In [ ]:
single_result, single_sampler = run_iterative(3, 1793)
print("epsilon:", 0.03, "alpha:", 0.01)
for row in single_result.iterations:
    print("Grover power, shots, successes:", row.k, row.shots, row.successes)


## Probability confidence interval
Source code cell 19.


In [ ]:
print(single_result.confidence_interval)
assert (
    single_result.confidence_interval[0]
    <= grid_pi(8) / 4
    <= single_result.confidence_interval[1]
)


## Pi from the iterative probability estimate
Source code cell 20.


In [ ]:
print("Iterative pi:", 4 * single_result.estimate)
print(
    "This interval covers finite-grid probability, not continuum discretization error."
)


## All four original lattice widths
Source code cell 23.


In [ ]:
runs = tuple(run_iterative(bits, 1800 + bits) for bits in range(1, 5))
results = tuple(row for row, _ in runs)
for bits, (result, sampler) in enumerate(runs, 1):
    print(
        "Coordinate bits:",
        bits,
        "pi:",
        4 * result.estimate,
        "probability interval:",
        result.confidence_interval,
    )
    print(
        "Actual shots:",
        sum(row.shots for row in result.iterations),
        "visited powers:",
        tuple(sampler.checks),
    )
    assert (
        result.confidence_interval[0]
        <= grid_pi(1 << bits) / 4
        <= result.confidence_interval[1]
    )
    for check in sampler.checks.values():
        print(check)


## Measured values replace source commented samples
Source code cell 25.


In [ ]:
print("Local estimates:", [4 * result.estimate for result in results])
print("Local intervals:", [result.confidence_interval for result in results])
print("Original commented outputs are illustrative hosted runs, not local evidence.")


## Finite-grid and sampling comparison
Source code cell 26.


In [ ]:
show_figure(plot_estimates(results))
print(
    "Horizontal axis is lattice size; actual quantum observation counts are printed separately."
)
